In [1]:
import os, sys
from pathlib import Path

# Localizar a raiz do projeto (pasta que contém "src/") para poder importar os módulos auxiliares
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
print("Raiz do projeto:", PROJECT_ROOT)

Raiz do projeto: /mnt/attach/outputs/IA3_Projeto-Creditcard


# 2. Data Preparation (Metodologia CRISP-ML)

## 2.1. Objetivos
Limpar o dataset, criar variáveis pré-transação, dividir **cronologicamente** em treino (80% mais antigo) e teste (20% mais recente)
e ajustar o `StandardScaler` **apenas no treino** (evita *data leakage*). Os resultados são guardados em `data/processed/`.

## 2.2. Limpeza e ordenação cronológica

In [2]:
import pandas as pd
from src import data_preparation as dp
from src.config import RAW_FILE

df_bruto = pd.read_csv(RAW_FILE)
df_limpo = dp.limpar_dados(df_bruto)

[LIMPEZA CONCLUÍDA] 284807 -> 283726 registos (1081 duplicados removidos)


## 2.3. Engenharia de funcionalidades
- `Hour`: hora do dia (0–23) derivada de `Time`.
- `Amount_log`: `log1p(Amount)` para reduzir a assimetria.

In [3]:
df_feat = dp.criar_features(df_limpo)
df_feat[["Time", "Hour", "Amount", "Amount_log"]].describe().round(3)

,Time,Hour,Amount,Amount_log
count,283726.000,283726.000,283726.000,283726.000
mean,94811.078,14.046,88.473,3.154
std,47481.048,5.835,250.399,1.657
min,0.000,0.000,0.000,0.000
25%,54204.750,10.000,5.600,1.887
50%,84692.500,15.000,22.000,3.135
75%,139298.000,19.000,77.510,4.363
max,172792.000,23.000,25691.160,10.154


## 2.4. Divisão temporal treino/teste

In [4]:
treino, teste = dp.dividir_temporal(df_feat)

Treino: 226980 (399 fraudes) | Teste: 56746 (74 fraudes)


## 2.5. Tratamento de nulos (defensivo) e normalização (StandardScaler só no treino)

In [5]:
treino, teste = dp.tratar_nulos(treino, teste)
scaler = dp.ajustar_scaler(treino)
from src.config import FEATURES
pd.DataFrame(scaler.transform(treino[FEATURES]), columns=FEATURES).describe().round(3).loc[["mean", "std"]].iloc[:, :8]

,V1,V2,V3,V4,V5,V6,V7,V8
mean,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 2.6. Pipeline de execução e exportação final

In [6]:
tr, te, scaler = dp.preparar_pipeline()
print(f"\nDimensões -> treino {tr.shape} | teste {te.shape}")
print("Fraudes no treino:", int(tr["Class"].sum()), "| no teste:", int(te["Class"].sum()))

[LIMPEZA CONCLUÍDA] 284807 -> 283726 registos (1081 duplicados removidos)
Treino: 226980 (399 fraudes) | Teste: 56746 (74 fraudes)


Guardado: creditcard_train_prepared.csv, creditcard_test_prepared.csv, scaler.joblib, features_list.joblib

Dimensões -> treino (226980, 32) | teste (56746, 32)
Fraudes no treino: 399 | no teste: 74


**Nota:** o desequilíbrio **não** é corrigido aqui; é tratado na modelação (`class_weight`, amostras) para manter o teste realista.